In [4]:
import os
import orjson
from collections import Counter
from tqdm.auto import tqdm # Automatically chooses the best progress bar for Notebooks

# Define your exact paths based on the VS Code directory tree
TRAIN_ANNOS_DIR = "train/annos"
TRAIN_IMAGE_DIR = "train/image"
VAL_ANNOS_DIR = "validation/annos"
VAL_IMAGE_DIR = "validation/image"

# --- 1. Helper Functions ---
def get_image_set(image_dir):
    """Loads all image filenames into RAM for instant O(1) lookup."""
    print(f"🔍 Scanning image directory: {image_dir}...")
    return {f for f in os.listdir(image_dir) if f.lower().endswith('.jpg')}

def calculate_class_weights(processed_dataset, top_5_ids):
    """Calculates weights to handle class imbalance using inverse frequencies."""
    total_valid_items = sum([len(img["item_categories"]) for img in processed_dataset])
    class_weights = {}
    
    for cat_id in top_5_ids:
        cat_count = sum(1 for img in processed_dataset for c in img["item_categories"] if c == cat_id)
        # Inverse frequency: Total / (Num_Classes * Count)
        weight = total_valid_items / (len(top_5_ids) * cat_count) if cat_count > 0 else 0
        class_weights[str(cat_id)] = weight
        
    return class_weights

def process_directory(json_files, image_dir, top_5, desc="Processing"):
    """Extracts bounding boxes, labels, and polygons using a linear, Notebook-safe loop."""
    available_images = get_image_set(image_dir)
    processed_dataset = []
    
    for filepath in tqdm(json_files, desc=desc):
        filename = os.path.basename(filepath)
        image_filename = filename.replace('.json', '.jpg')
        
        # Fast Set Lookup (replaces the slow os.path.exists)
        if image_filename not in available_images:
            continue
            
        try:
            with open(filepath, 'rb') as f:
                data = orjson.loads(f.read())
        except Exception:
            continue
            
        image_data = {
            "image_path": os.path.join(image_dir, image_filename).replace("\\", "/"),
            "classification_labels": set(),
            "detection_bboxes": [],
            "segmentation_polygons": [],
            "item_categories": [] 
        }
        
        for key, value in data.items():
            if key.startswith("item") and isinstance(value, dict):
                cat_id = value.get("category_id")
                
                if cat_id in top_5:
                    image_data["classification_labels"].add(cat_id)
                    image_data["detection_bboxes"].append(value.get("bounding_box"))
                    image_data["segmentation_polygons"].append(value.get("segmentation"))
                    image_data["item_categories"].append(cat_id)
                    
        # Only keep if it has a valid top-5 item
        if image_data["classification_labels"]:
            image_data["classification_labels"] = list(image_data["classification_labels"])
            processed_dataset.append(image_data)
            
    return processed_dataset


# ==========================================
# Execution Workflow
# ==========================================
if __name__ == '__main__':
    # 1. Identify Top 5 Categories
    print("1. Identifying Top 5 Categories from Training Data...")
    train_files = [os.path.join(TRAIN_ANNOS_DIR, f) for f in os.listdir(TRAIN_ANNOS_DIR) if f.endswith(".json")]
    
    total_counts = Counter()
    for filepath in tqdm(train_files, desc="Counting Categories"):
        try:
            with open(filepath, 'rb') as f:
                data = orjson.loads(f.read())
            for key, value in data.items():
                if key.startswith("item") and isinstance(value, dict):
                    cat_id = value.get("category_id")
                    if cat_id is not None:
                        total_counts[cat_id] += 1
        except Exception:
            continue

    top_5_ids = [cat_id for cat_id, count in total_counts.most_common(5)]
    print(f"✅ Top 5 Category IDs selected: {top_5_ids}")

    # 2. Process Training Set
    print("\n2. Processing Training Set...")
    train_data = process_directory(train_files, TRAIN_IMAGE_DIR, top_5_ids, desc="Train Set")
    print(f"✅ Total training images kept: {len(train_data)}")

    # 3. Process Validation Set
    print("\n3. Processing Validation Set...")
    val_files = [os.path.join(VAL_ANNOS_DIR, f) for f in os.listdir(VAL_ANNOS_DIR) if f.endswith(".json")]
    val_data = process_directory(val_files, VAL_IMAGE_DIR, top_5_ids, desc="Val Set")
    print(f"✅ Total validation images kept: {len(val_data)}")

    # 4. Calculate Weights
    print("\n4. Calculating Class Weights for Imbalance...")
    weights = calculate_class_weights(train_data, top_5_ids)
    print(f"✅ Weights for Loss Function: {weights}")

    # 5. Save to Disk
    print("\n5. Saving Results to JSON...")
    with open("processed_train_data.json", "wb") as f:
        f.write(orjson.dumps({
            "top_5_categories": top_5_ids, 
            "class_weights": weights, 
            "data": train_data
        }))
        
    with open("processed_val_data.json", "wb") as f:
        f.write(orjson.dumps({
            "top_5_categories": top_5_ids, 
            "data": val_data
        }))

    print("Preprocessing Complete!")

1. Identifying Top 5 Categories from Training Data...


Counting Categories: 100%|██████████| 191961/191961 [00:43<00:00, 4383.99it/s]


✅ Top 5 Category IDs selected: [1, 8, 7, 2, 9]

2. Processing Training Set...
🔍 Scanning image directory: train/image...


Train Set: 100%|██████████| 191961/191961 [00:59<00:00, 3225.03it/s]


✅ Total training images kept: 144174

3. Processing Validation Set...
🔍 Scanning image directory: validation/image...


Val Set: 100%|██████████| 32153/32153 [00:17<00:00, 1822.31it/s]


✅ Total validation images kept: 23741

4. Calculating Class Weights for Imbalance...
✅ Weights for Loss Function: {'1': 0.6435815479098332, '8': 0.8324949897990503, '7': 1.25926917194669, '2': 1.2785437000887312, '9': 1.4953591697746067}

5. Saving Results to JSON...
Preprocessing Complete!


In [5]:
import json
from collections import Counter
import os

def print_class_distribution(json_path):
    if not os.path.exists(json_path):
        print(f"File not found: {json_path}")
        return

    print(f"\n📊 Class Distribution for: {os.path.basename(json_path)}")
    
    with open(json_path, 'r') as f:
        full_json = json.load(f)
        data = full_json["data"]

    # DeepFashion2 Top 5 Mapping
    category_names = {
        1: "Short Sleeve Top",
        8: "Trousers",
        7: "Shorts",
        2: "Long Sleeve Top",
        9: "Skirt"
    }

    # Extract all category IDs from all images
    all_categories = []
    for item in data:
        all_categories.extend(item.get("item_categories", []))

    # Count them up
    category_counts = Counter(all_categories)
    total_instances = sum(category_counts.values())

    print(f"Total Images: {len(data)}")
    print(f"Total Clothing Instances: {total_instances}")
    print("-" * 65)
    print(f"{'Cat ID':<8} | {'Category Name':<20} | {'Count':<10} | {'Percentage'}")
    print("-" * 65)

    # Print sorted by frequency
    for cat_id, count in category_counts.most_common():
        name = category_names.get(cat_id, "Unknown")
        percentage = (count / total_instances) * 100
        print(f"{cat_id:<8} | {name:<20} | {count:<10} | {percentage:.2f}%")
    print("-" * 65)

# ==========================================
# Execution
# ==========================================
# Point these to the NEW json files created by your trimming script
NEW_TRAIN_JSON = r"./processed_train_data.json"
NEW_VAL_JSON = r"./processed_val_data.json"

print_class_distribution(NEW_TRAIN_JSON)
print_class_distribution(NEW_VAL_JSON)


📊 Class Distribution for: processed_train_data.json
Total Images: 144174
Total Clothing Instances: 230547
-----------------------------------------------------------------
Cat ID   | Category Name        | Count      | Percentage
-----------------------------------------------------------------
1        | Short Sleeve Top     | 71645      | 31.08%
8        | Trousers             | 55387      | 24.02%
7        | Shorts               | 36616      | 15.88%
2        | Long Sleeve Top      | 36064      | 15.64%
9        | Skirt                | 30835      | 13.37%
-----------------------------------------------------------------

📊 Class Distribution for: processed_val_data.json
Total Images: 23741
Total Clothing Instances: 38797
-----------------------------------------------------------------
Cat ID   | Category Name        | Count      | Percentage
-----------------------------------------------------------------
1        | Short Sleeve Top     | 12556      | 32.36%
8        | Trousers 